# Experiment with Overlapping Chunks

<cite id="1pwz6"><a href="#zotero%7C10812%2FFP4M5PFQ">(Recinos, 1947)</a></cite>


In [11]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from ipywidgets import interact
from IPython.display import HTML

In [2]:
def import_tokens(src_id):
    src_path = f"../{src_id}"
    token_file = f"{src_path}/{src_id}-TOKEN.csv"
    TOKEN = pd.read_csv(token_file)
    idx_offset = TOKEN.columns.to_list().index('token_str')
    ohco = TOKEN.columns.to_list()[:idx_offset]
    return TOKEN.set_index(ohco)

def chunk_tokens(TOKEN, chunk_size=150, overlap=20, min_len=50):
    """Split text into overlapping word-level chunks."""
    tokens = TOKEN.term_str.dropna().to_list()
    chunks = []
    for i in range(0, len(tokens), chunk_size - overlap):
        chunk = " ".join(tokens[i:i + chunk_size])
        if len(chunk.split()) >= min_len:  # drop tiny tail chunks
            chunks.append(chunk)
    return chunks

def get_nmf_topics(nmf_model, tfidf_vectorizer, n_top_words):
    words = tfidf_vectorizer.get_feature_names_out()
    topic_words = {}
    for topic_idx, topic in enumerate(nmf_model.components_):
        top_word_indices = topic.argsort()[: -n_top_words - 1 : -1]
        topic_words[f"Topic {topic_idx}"] = [words[i] for i in top_word_indices]    
    return pd.DataFrame(topic_words)

In [3]:
sources = {
    'quc': ['ajtzibab', 'christenson', 'colop', 'christenson_ximenez','ximenez'],
    'spa': ['recinos'],
    'eng': ['tedlock']
}
SOURCES = {}
for lang in sources:
    for src_id in sources[lang]:
        SOURCES[src_id] = {}
        SOURCES[src_id]['lang'] = lang
        SOURCES[src_id]['label'] = src_id.replace("_", " ").title()
        SOURCES[src_id]['tokens'] = import_tokens(src_id)
        # SOURCES[src_id]['text_len'] = SOURCES[src_id]['tokens'].shape[0]

In [16]:
@interact(
    min_df = (1, 20, 1),
    max_df = (.1, 1, .01),
    src_id = SOURCES.keys(), 
    n_topics = (2, 20, 1),
    chunk_size = (100, 2000, 5), 
    overlap = (0., .9, .01)
)
def plot_text(
        src_id = 'colop', 
        chunk_size = 1000, 
        overlap = .9,
        min_df = 5,
        max_df = .35,
        n_topics = 8
    ):

    # Create chunked data
    overlap_int = int(overlap * chunk_size)
    TOKEN = SOURCES[src_id]['tokens']
    chunks = chunk_tokens(TOKEN, chunk_size=chunk_size, overlap=overlap_int)

    # Create Count matrix
    count_engine = TfidfVectorizer(lowercase=True, 
        max_df=max_df, 
        min_df=min_df, 
        strip_accents=None,
        norm='l2')
    X = count_engine.fit_transform(chunks)

    # Create topic model
    topic_engine = NMF(
        n_components=n_topics,
        init='nndsvd',
        max_iter=500
    )
    global THETA
    THETA = pd.DataFrame(topic_engine.fit_transform(X))
    THETA.index.name = 'chunk_id'
    THETA.columns.name = 'topic_id'

    # Get topics
    global TOPICS
    TOPICS = get_nmf_topics(topic_engine, count_engine, 7)
    
    # Plot heatmap
    fig, ax = plt.subplots(figsize=(20,4))
    sns.heatmap(THETA.T, cmap="YlGnBu")
    plt.title(f"{src_id.replace('_', ' ').title()}", fontdict={'size':20, 'weight':'bold'}, y=1.01)
    plt.xlabel("Syntagm / Event", fontdict={'size': 16})
    plt.ylabel("Paradigm / Structure", fontdict={'size': 16})
    plt.show()

    # Print table of topics
    # hr = "-" * 58
    # print(hr)
    # print("Text length:", len(TOKEN), "tokens")
    # print(hr)
    # print(TOPICS.apply(lambda x: ' '.join(x)))
    # print(hr)
    topic_table = TOPICS.apply(lambda x: ' '.join(x)).to_frame('top_terms').to_html()
    display(HTML(topic_table))

    # Plot lines graphs
    # axes = THETA.plot.area(subplots=True, alpha=.25, sharex=True, sharey=True, legend=False, figsize=(9.5, TOPICS.shape[1]*1.5))
    # for i, col in enumerate(THETA.columns):
    #     col_name = f"Topic {i}"
    #     axes[i].set_title(f"{col_name}: " + " ".join(TOPICS[col_name].to_list()))
    # sns.despine(left=True, bottom=True)
    # plt.xlabel("")
    # # plt.xticks([])
    # plt.yticks([])
    # plt.tight_layout()
    # plt.show()

interactive(children=(Dropdown(description='src_id', index=2, options=('ajtzibab', 'christenson', 'colop', 'ch…

## Bibliography

<!-- BIBLIOGRAPHY START -->
<div class="csl-bib-body">
  <div class="csl-entry"><i id="zotero|10812/FP4M5PFQ"></i>Recinos, A. (1947). <i>Popol Vuh: Las Antiguas Historias Del Quiché</i>. Fondo de Cultura Económica. <a href="https://books.google.com/books?hl=en&#38;lr=&#38;id=p9hpEAAAQBAJ&#38;oi=fnd&#38;pg=PT4&#38;dq=recinos+1947&#38;ots=B3jMvJjNMN&#38;sig=bwZONIo-kkQvOIMkwDBJjPoYzUQ">https://books.google.com/books?hl=en&#38;lr=&#38;id=p9hpEAAAQBAJ&#38;oi=fnd&#38;pg=PT4&#38;dq=recinos+1947&#38;ots=B3jMvJjNMN&#38;sig=bwZONIo-kkQvOIMkwDBJjPoYzUQ</a></div>
</div>
<!-- BIBLIOGRAPHY END -->